In [ ]:
"""Figure 6E–F: synapse proximity to tracheole terminal ends.

Inputs
------
Corrected CATMAID node table for 117 tracheoles (1,376 distal ends),
FAFB v783 Princeton presynaptic-contact table, and FlyWire neuron NT labels.

Analysis
--------
Figure 6E: all presynaptic contacts to their nearest distal terminal end.
Figure 6F: the same contact distances stratified by NT score >= 0.5.
Distances are Euclidean in FAFB nanometres, converted to micrometres.
Synapses are streamed in chunks, never loaded into memory all at once.
The 0.05-µm histogram resolves ECDFs and medians; the NT chi-square test
uses 1-µm bins and pools distances >= 50 µm into one final bin.
"""

from pathlib import Path
import json
import math
import re

import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from scipy.stats import chi2_contingency


# Input files and settings.
data_dir = Path("data")
node_csv = data_dir / "catmaid_nodes.csv.gz"
syn_csv = data_dir / "fafb_v783_princeton_synapse_table.csv"
nt_csv = data_dir / "neurons.csv"
out_dir = Path("fig4_outputs")

expected_tracheoles, expected_ends = 117, 1376
spacing_nm, nt_min_score = 100.0, 0.5
chunk_size, workers = 250_000, -1
hist_step, hist_max = 0.05, 100.0
fine_edges = np.arange(0.0, hist_max + hist_step * 0.5, hist_step)
hist_edges = np.append(fine_edges, np.inf)
n_hist_bins = len(hist_edges) - 1

nt_order = ["ACh", "GABA", "GLUT", "DA", "OCT", "SER"]
nt_colors = {"ACh": "#2020df", "GABA": "#df2020", "GLUT": "#df9c20",
             "DA": "#963939", "OCT": "#107010", "SER": "#701070"}

syn_root_cols = ["pre_root_id_720575940", "pre_root_id", "presynaptic_root_id", "pre_pt_root_id"]
syn_x_cols = ["pre_x", "presynaptic_x", "pre_pt_position_x"]
syn_y_cols = ["pre_y", "presynaptic_y", "pre_pt_position_y"]
syn_z_cols = ["pre_z", "presynaptic_z", "pre_pt_position_z"]
nt_root_cols = ["root_id", "root", "flywire_root_id", "segment_id"]
nt_type_cols = ["nt_type", "transmitter", "neurotransmitter"]
nt_score_cols = ["nt_type_score", "transmitter_prob", "transmitter_probability", "nt_score"]
skid_cols = ["skeleton_id", "new_skeleton_id", "skid", "neuron"]
node_cols = ["node_id", "treenode_id", "id"]
parent_cols = ["parent_id", "parent_node_id", "parent"]
trach_x_cols = ["x_nm", "x"]
trach_y_cols = ["y_nm", "y"]
trach_z_cols = ["z_nm", "z"]


def normalise_name(value):
    return re.sub(r"[^a-z0-9]+", "_", str(value).strip().lower()).strip("_")


def resolve_column(columns, candidates):
    lookup = {normalise_name(c): c for c in columns}
    for candidate in candidates:
        if normalise_name(candidate) in lookup:
            return lookup[normalise_name(candidate)]
    raise KeyError(f"Missing required column. Tried {candidates}. Available: {list(columns)}")


def clean_text(series):
    return series.astype("string").str.strip().replace(
        {"": pd.NA, "nan": pd.NA, "None": pd.NA, "<NA>": pd.NA})


def clean_root_id(series):
    return clean_text(series).str.replace(r"\.0$", "", regex=True)


def nt_display_name(value):
    key = re.sub(r"[^a-z0-9]+", "", str(value).strip().lower())
    return {"ach": "ACh", "acetylcholine": "ACh", "gaba": "GABA",
            "glut": "GLUT", "glutamate": "GLUT", "da": "DA", "dopamine": "DA",
            "oct": "OCT", "octopamine": "OCT", "ser": "SER",
            "serotonin": "SER", "5ht": "SER"}.get(key)


def load_trachea_nodes(path):
    raw = pd.read_csv(path, low_memory=False)
    col = list(raw.columns)
    fields = [resolve_column(col, candidates) for candidates in
              (skid_cols, node_cols, parent_cols, trach_x_cols, trach_y_cols, trach_z_cols)]
    nodes = raw[fields].copy()
    nodes.columns = ["skeleton_id", "node_id", "parent_id", "x_nm", "y_nm", "z_nm"]
    for name in nodes.columns:
        nodes[name] = pd.to_numeric(nodes[name], errors="coerce")
    nodes = nodes.dropna(subset=["skeleton_id", "node_id", "x_nm", "y_nm", "z_nm"])
    nodes["skeleton_id"] = nodes["skeleton_id"].astype(np.int64)
    nodes["node_id"] = nodes["node_id"].astype(np.int64)
    nodes["parent_id"] = nodes["parent_id"].astype("Int64")
    return nodes.reset_index(drop=True)


def load_nt_lookup(path):
    header = pd.read_csv(path, nrows=0).columns
    root, nt, score = [resolve_column(header, names) for names in
                       (nt_root_cols, nt_type_cols, nt_score_cols)]
    table = pd.read_csv(path, usecols=[root, nt, score], dtype={root: "string"}, low_memory=False)
    table = table.rename(columns={root: "root_id", nt: "nt_type", score: "nt_type_score"})
    table["root_id"] = clean_root_id(table["root_id"])
    table["nt_type"] = clean_text(table["nt_type"])
    table["nt_type_score"] = pd.to_numeric(table["nt_type_score"], errors="coerce")
    table = table.dropna(subset=["root_id"])
    table = table.sort_values(["root_id", "nt_type_score"], ascending=[True, False],
                              na_position="last").drop_duplicates("root_id", keep="first")
    return table.reset_index(drop=True)


def tracheole_geometry(nodes):
    """Build the 100-nm centreline probe and distal-end coordinates."""
    sampled, terminal_rows = [], []
    for sid, group in nodes.groupby("skeleton_id", sort=False):
        ids = group["node_id"].to_numpy(np.int64)
        xyz = group[["x_nm", "y_nm", "z_nm"]].to_numpy(np.float64)
        lookup = {int(nid): xyz[i] for i, nid in enumerate(ids)}
        orphan = group.loc[group["parent_id"].notna() &
                           ~group["parent_id"].isin(group["node_id"]), "node_id"]
        parents = set(group["parent_id"].dropna().astype(np.int64))
        ends = group.loc[~group["node_id"].isin(parents) & group["parent_id"].notna()]
        sections = [group.loc[group["parent_id"].isna(),
                              ["x_nm", "y_nm", "z_nm"]].to_numpy(np.float64)]
        for row in group.itertuples(index=False):
            if pd.isna(row.parent_id):
                continue
            parent = lookup[int(row.parent_id)]
            child = np.array([row.x_nm, row.y_nm, row.z_nm], dtype=np.float64)
            vector = child - parent
            length = float(np.linalg.norm(vector))
            if length == 0.0:
                sections.append(child[None, :])
            else:
                n = max(int(math.ceil(length / spacing_nm)), 1)
                frac = np.arange(1, n + 1, dtype=np.float64) / n
                sections.append(parent[None, :] + frac[:, None] * vector[None, :])
        sampled.append(np.vstack(sections))
        for row in ends.itertuples(index=False):
            terminal_rows.append((int(sid), int(row.node_id),
                                  float(row.x_nm), float(row.y_nm), float(row.z_nm)))
    terminals = pd.DataFrame(terminal_rows, columns=["skeleton_id", "node_id", "x_nm", "y_nm", "z_nm"])
    return np.vstack(sampled).astype(np.float32, copy=False), terminals


def root_prefix(column_name):
    match = re.search(r"root_id_(\d{9})", str(column_name))
    return match.group(1) if match else None


def transform_roots(series, mode, prefix):
    cleaned = clean_root_id(series)
    if mode == "raw":
        return cleaned
    if mode == "header_prefix":
        suffix_len = 18 - len(prefix)
        short = cleaned.notna() & cleaned.str.fullmatch(rf"\d{{1,{suffix_len}}}")
        cleaned.loc[short] = prefix + cleaned.loc[short].str.zfill(suffix_len)
        return cleaned
    raise ValueError(mode)


def select_root_mode(path, root_col, known_roots):
    probe = pd.read_csv(path, usecols=[root_col], dtype={root_col: "string"}, nrows=20_000)[root_col]
    options = [("raw", None)]
    prefix = root_prefix(root_col)
    if prefix:
        options.append(("header_prefix", prefix))
    def match_score(option):
        roots = set(transform_roots(probe, *option).dropna().astype(str))
        return len(roots & known_roots) / len(roots) if roots else 0.0
    best = max(options, key=match_score)
    score = match_score(best)
    return best, score


def transform_xyz(xyz, mode):
    if mode == "nm":
        return np.asarray(xyz, dtype=np.float64)
    if mode == "voxel_4_4_40":
        return np.asarray(xyz, dtype=np.float64) * [4.0, 4.0, 40.0]


def select_coordinate_mode(path, xyz_cols, trachea_points, trachea_tree):
    probe = pd.read_csv(path, usecols=xyz_cols, nrows=20_000, low_memory=False)
    xyz = probe[xyz_cols].apply(pd.to_numeric, errors="coerce").dropna().to_numpy(np.float64))
    lo = np.min(trachea_points, axis=0) - 50_000.0
    hi = np.max(trachea_points, axis=0) + 50_000.0
    options = []
    for mode in ("nm", "voxel_4_4_40"):
        candidate = transform_xyz(xyz, mode)
        within = np.all((candidate >= lo) & (candidate <= hi), axis=1).mean()
        near, _ = trachea_tree.query(candidate.astype(np.float32), workers=workers)
        options.append((mode, float(within), float(np.median(near) / 1000.0)))
    best = min(options, key=lambda item: (-item[1], item[2]))
    return best[0], options


def empty_hist():
    return np.zeros(n_hist_bins, dtype=np.int64)


def add_hist(hist, data):
    data = np.asarray(data, dtype=np.float64)
    data = data[np.isfinite(data)]
    if len(data):
        hist += np.histogram(data, bins=hist_edges)[0]


def hist_quantile(hist, q):
    total = int(np.sum(hist))
    if not total:
        return float("nan")
    counts = np.cumsum(hist)
    index = int(np.searchsorted(counts, q * total, side="left"))
    if index >= len(fine_edges) - 1:
        return float("nan")
    prev = int(counts[index - 1]) if index else 0
    count = int(hist[index])
    return float(fine_edges[index] + (q * total - prev) / count * hist_step)


def ecdf_from_hist(hist):
    total = int(hist.sum())
    if not total:
        return np.array([]), np.array([])
    return fine_edges[1:], np.cumsum(hist[:-1]) / total


def stream_contact_histograms(syn_path, nt_lookup, trachea_points, trachea_tree, end_tree):
    columns = pd.read_csv(syn_path, nrows=0).columns
    root = resolve_column(columns, syn_root_cols)
    xyz_cols = [resolve_column(columns, candidates) for candidates in
                (syn_x_cols, syn_y_cols, syn_z_cols)]
    (root_mode, prefix), match_fraction = select_root_mode(
        syn_path, root, set(nt_lookup["root_id"].dropna().astype(str)))
    coord_mode, coord_diagnostics = select_coordinate_mode(
        syn_path, xyz_cols, trachea_points, trachea_tree)

    nt_index = nt_lookup.set_index("root_id")[["nt_type", "nt_type_score"]]
    all_hist = empty_hist()
    nt_hist = {name: empty_hist() for name in nt_order}
    all_rows, valid_contacts, assigned_contacts = 0, 0, 0
    reader = pd.read_csv(syn_path, usecols=[root, *xyz_cols], dtype={root: "string"},
                         chunksize=chunk_size, low_memory=False)
    for chunk in reader:
        all_rows += len(chunk)
        xyz = chunk[xyz_cols].apply(pd.to_numeric, errors="coerce").to_numpy(np.float64)
        valid = np.all(np.isfinite(xyz), axis=1)
        if not valid.any():
            continue
        coords = transform_xyz(xyz[valid], coord_mode).astype(np.float32)
        distances_nm, _ = end_tree.query(coords, workers=workers)
        distances_um = distances_nm / 1000.0
        add_hist(all_hist, distances_um)
        valid_contacts += len(distances_um)

        root_values = transform_roots(chunk.loc[valid, root], root_mode, prefix).reset_index(drop=True)
        joined = pd.DataFrame({"root_id": root_values}).join(nt_index, on="root_id")
        high = (joined["nt_type"].notna() &
                joined["nt_type_score"].ge(nt_min_score)).fillna(False).to_numpy(bool)
        if high.any():
            canonical = np.array([nt_display_name(x) for x in joined.loc[high, "nt_type"]], dtype=object)
            distances_high = distances_um[high]
            for name in nt_order:
                mask = canonical == name
                if mask.any():
                    add_hist(nt_hist[name], distances_high[mask])
                    assigned_contacts += int(mask.sum())
    print(f"Princeton rows: {all_rows:,}; valid presynaptic contacts: {valid_contacts:,}; six-NT contacts: {assigned_contacts:,}")
    return all_hist, nt_hist, {
        "n_source_rows": all_rows, "n_valid_contacts": valid_contacts,
        "n_six_nt_contacts": assigned_contacts, "nt_threshold": nt_min_score,
        "coordinate_mode": coord_mode, "coordinate_probe": coord_diagnostics,
        "root_id_mode": root_mode, "root_id_match_fraction": match_fraction,
    }


def rebin_for_test(hist):
    """0.05-µm bins to [0,1), ..., [49,50), [50,infinity)."""
    counts = np.asarray(hist, dtype=np.int64)[:-1]
    index = np.floor(fine_edges[:-1]).astype(int)
    regular = (index >= 0) & (index < 50)
    result = np.zeros(51, dtype=np.int64)
    np.add.at(result, index[regular], counts[regular])
    result[-1] = counts[~regular].sum() + int(hist[-1])
    return result


def style_axes(ax):
    ax.set(xlim=(0, 45), ylim=(0, 1.01),
           xlabel="Synapse-to-nearest-terminal-end distance (µm)",
           ylabel="Cumulative fraction")
    ax.xaxis.labelpad = ax.yaxis.labelpad = 7
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_linewidth(1.0)
    ax.tick_params(axis="both", which="major", direction="out")
    ax.set_xticks([0, 15, 30, 45])
    ax.xaxis.set_minor_locator(MultipleLocator(5))
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(["0", "", "", "", "1"])
    ax.grid(False)
    ax.figure.subplots_adjust(left=0.17, right=0.975, bottom=0.17, top=0.965)


def save_figure(fig, stem):
    for extension in ("svg", "pdf", "png"):
        fig.savefig(out_dir / f"{stem}.{extension}", dpi=600,
                    bbox_inches="tight", facecolor="white")


def main():
    out_dir.mkdir(parents=True, exist_ok=True)
    nodes = load_trachea_nodes(node_csv)
    trachea_points, ends = tracheole_geometry(nodes)
    trachea_tree = cKDTree(trachea_points)
    end_tree = cKDTree(ends[["x_nm", "y_nm", "z_nm"]].to_numpy(np.float32))
    all_hist, nt_hist, metadata = stream_contact_histograms(
        syn_csv, load_nt_lookup(nt_csv), trachea_points, trachea_tree, end_tree)
    metadata.update({"n_tracheoles": expected_tracheoles, "n_terminal_ends": len(ends),
                     "corrected_node_file": node_csv.name, "hist_step_um": hist_step,
                     "hist_max_um": hist_max, "chi_square_bin_um": 1.0,
                     "chi_square_tail_ge_um": 50.0})

    plt.rcParams.update({
        "font.family": "Arial", "font.size": 12, "axes.labelsize": 13,
        "xtick.labelsize": 11.5, "ytick.labelsize": 11.5,
        "axes.linewidth": 1.0, "xtick.major.width": 1.0, "ytick.major.width": 1.0,
        "xtick.major.size": 5, "ytick.major.size": 5,
        "xtick.direction": "out", "ytick.direction": "out",
        "svg.fonttype": "none", "pdf.fonttype": 42, "ps.fonttype": 42,
    })

    median_all = hist_quantile(all_hist, 0.5)
    x, y = ecdf_from_hist(all_hist)
    fig, ax = plt.subplots(figsize=(5.7, 4.8), facecolor="white")
    ax.plot(x, y, color="#303030", linewidth=2.0, solid_capstyle="round", zorder=3)
    ax.axvline(median_all, color="#777777", linestyle=(0, (3, 2.5)),
               linewidth=1.25, zorder=2)
    ax.text(median_all, 0.955, f"{median_all:.2f} µm", ha="center", va="bottom",
            fontsize=12, color="#4D4D4D")
    style_axes(ax)
    save_figure(fig, "fig6e")
    plt.show()
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(5.7, 4.8), facecolor="white")
    for name in nt_order:
        x, y = ecdf_from_hist(nt_hist[name])
        ax.plot(x, y, color=nt_colors[name], linewidth=1.45,
                solid_capstyle="round", label=name)
    style_axes(ax)
    ax.legend(frameon=False, loc="upper left", fontsize=8.5,
              handlelength=1.8, labelspacing=0.25, borderaxespad=0.5)
    save_figure(fig, "fig6f")
    plt.show()
    plt.close(fig)

    contingency = np.vstack([rebin_for_test(nt_hist[name]) for name in nt_order])
    contingency = contingency[:, contingency.sum(axis=0) > 0]
    chi2, p, df, _ = chi2_contingency(contingency, correction=False)
    n_nt = float(contingency.sum())
    cramers_v = float(np.sqrt(chi2 / (n_nt * min(contingency.shape[0] - 1,
                                                contingency.shape[1] - 1))))
    summary = pd.DataFrame([
        {"nt_type": name, "n_contacts": int(nt_hist[name].sum()),
         "median_um": hist_quantile(nt_hist[name], 0.5),
         "q25_um": hist_quantile(nt_hist[name], 0.25),
         "q75_um": hist_quantile(nt_hist[name], 0.75)}
        for name in nt_order
    ])
    stats = pd.DataFrame([{"n_tracheoles": expected_tracheoles, "n_terminal_ends": len(ends),
                           "n_all_contacts": int(all_hist.sum()), "all_contacts_median_um": median_all,
                           "n_nt_contacts": int(n_nt), "chi_square": chi2, "df": int(df),
                           "p_value": float(p), "cramers_v": cramers_v}])
    ends.to_csv(out_dir / "fig4_terminal_ends_corrected.csv", index=False)
    summary.to_csv(out_dir / "fig4_nt_summary.csv", index=False)
    stats.to_csv(out_dir / "fig4_stats.csv", index=False)
    np.savez_compressed(out_dir / "fig4_histograms_CORRECTED_1376.npz",
                        finite_edges_um=fine_edges, all_terminal_ends=all_hist,
                        **{f"nt_{name}_terminal_ends": nt_hist[name] for name in nt_order})
    (out_dir / "fig4_run_metadata.json").write_text(json.dumps(metadata, indent=2))

    print(f"Tracheoles: {expected_tracheoles}; terminal ends: {len(ends):,}")
    print(f"Figure 6E: all-contact median = {median_all:.3f} µm (n = {int(all_hist.sum()):,})")
    print("Figure 6F NT medians (µm):")
    print(summary.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
    print(f"Figure 6F: χ²({df}) = {chi2:,.3f}, P = {p:.6g}, Cramer's V = {cramers_v:.6f}")

if __name__ == "__main__":
    main()